# Notebook 3: Advanced RAG Patterns & Agentic Behavior

**Duration:** 120 minutes  
**Level:** Advanced  

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Implement agentic RAG with conditional routing
- ✅ Build fallback mechanisms using web search
- ✅ Create self-correcting RAG pipelines
- ✅ Implement conversational RAG with memory
- ✅ Design adaptive retrieval strategies
- ✅ Evaluate agentic RAG systems effectively

---

## Table of Contents

1. [Introduction to Agentic RAG](#1-introduction-to-agentic-rag)
2. [Environment Setup](#2-environment-setup)
3. [Conditional Routing](#3-conditional-routing)
4. [Web Search Fallback](#4-web-search-fallback)
5. [Self-Correcting RAG (Self-RAG)](#5-self-correcting-rag)
6. [Conversational RAG with Memory](#6-conversational-rag-with-memory)
7. [Adaptive Retrieval Strategies](#7-adaptive-retrieval-strategies)
8. [Comprehensive Agentic Pipeline](#8-comprehensive-agentic-pipeline)
9. [Evaluation & Metrics](#9-evaluation-metrics)
10. [Exercises & Challenges](#10-exercises-challenges)
11. [Summary & Next Steps](#11-summary-next-steps)

---

## 1. Introduction to Agentic RAG

### What Makes RAG "Agentic"?

Traditional RAG follows a fixed pipeline:
```
Query → Retrieve → Generate → Done
```

**Agentic RAG** makes intelligent decisions:
```
Query → Analyze → Route to appropriate source
              → Validate quality
              → Self-correct if needed
              → Remember context
              → Generate → Done
```

### Key Agentic Capabilities

| Capability | Description | Benefit |
|------------|-------------|----------|
| **Conditional Routing** | Choose different paths based on query type | Optimized processing |
| **Web Fallback** | Search web when local docs insufficient | Extended knowledge |
| **Self-Correction** | Validate and improve responses | Higher accuracy |
| **Memory** | Track conversation history | Natural dialogue |
| **Adaptive Retrieval** | Adjust strategy based on complexity | Efficiency + quality |

### Evolution of RAG Systems

**Basic RAG (Notebooks 1-2):**
- Static pipeline
- One retrieval strategy
- No validation
- Stateless

**Agentic RAG (This notebook):**
- Dynamic decision-making
- Multiple strategies
- Self-validation
- Stateful conversations

## 2. Environment Setup

In [1]:
# Standard library
import os
import json
import time
import warnings
from typing import List, Dict, Any, Optional, Tuple
from collections import deque

# Third-party
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack core
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import ChatMessage

# Haystack components
from haystack.components.builders import PromptBuilder, ChatPromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever, InMemoryEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.joiners import DocumentJoiner

# Haystack routing and conditional logic
from haystack.components.routers import ConditionalRouter

# Web search
from haystack.components.websearch import SerperDevWebSearch
from haystack.components.converters import OutputAdapter

warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


In [2]:
# Load environment
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")  # For web search

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY required")

print("✅ Environment configured")
print(f"OpenAI Key: {OPENAI_API_KEY[:20]}...")
if SERPER_API_KEY:
    print(f"Serper Key: {SERPER_API_KEY[:20]}...")
else:
    print("⚠️  SerperDev API key not found (needed for web fallback example)")
    print("   Get free key at: https://serper.dev")

✅ Environment configured
OpenAI Key: sk-proj-DKrbGLW_syHN...
Serper Key: 18b752e22c064767d250...


## 3. Conditional Routing

Conditional routing directs queries to different pipeline branches based on conditions.

### 3.1 Understanding Conditional Router

The `ConditionalRouter` evaluates conditions and routes to appropriate outputs.

In [3]:
# Create sample data
def create_company_kb() -> List[Document]:
    """Create sample company knowledge base."""
    docs_data = [
        {"title": "Leave Policy", "content": "Employees get 20 days annual leave. Sick leave is 10 days per year."},
        {"title": "Remote Work", "content": "Remote work allowed up to 3 days/week with manager approval."},
        {"title": "Benefits", "content": "Health insurance covers employee and family. Gym membership included."},
        {"title": "Working Hours", "content": "Standard hours: 9 AM - 5 PM. Flexible hours available."},
        {"title": "Expenses", "content": "Business expenses reimbursed within 30 days. Keep all receipts."},
    ]
    return [Document(content=d["content"], meta={"title": d["title"]}) for d in docs_data]

company_docs = create_company_kb()
company_store = InMemoryDocumentStore()
company_store.write_documents(company_docs)

print(f"✅ Created knowledge base with {len(company_docs)} documents")

✅ Created knowledge base with 5 documents


In [5]:
routing_rules = [
    {
        "condition": "'leave' in query or 'vacation' in query or 'sick' in query",  # ✅ plain Python
        "output": "{{ query }}",   # ✅ Jinja2 for output
        "output_name": "hr_query",
        "output_type": str,
    },
    {
        "condition": "'remote' in query or 'work from home' in query",  # ✅
        "output": "{{ query }}",
        "output_name": "remote_query",
        "output_type": str,
    },
    {
        "condition": "True",  # ✅ default fallback
        "output": "{{ query }}",
        "output_name": "general_query",
        "output_type": str,
    },
]

In [8]:
# Define routing rules
# routing_rules = [
#     {
#         "condition": "{{ 'leave' in query or 'vacation' in query or 'sick' in query }}",
#         "output": "{{ query }}",
#         "output_name": "hr_query",
#         "output_type": str,
#     },
#     {
#         "condition": "{{ 'remote' in query or 'work from home' in query }}",
#         "output": "{{ query }}",
#         "output_name": "remote_query",
#         "output_type": str,
#     },
#     {
#         "condition": "{{ query }}",  # Default fallback
#         "output": "{{ query }}",
#         "output_name": "general_query",
#         "output_type": str,
#     },
# ]

# Create router
router = ConditionalRouter(routes=routing_rules, unsafe=True)

# Test routing
test_queries = [
    "How many vacation days do I get?",
    "Can I work remotely?",
    "What are the working hours?"
]

print("\nTesting Query Router:\n")
for query in test_queries:
    result = router.run(query=query)
    route_taken = list(result.keys())[0]
    print(f"Query: {query}")
    print(f"  → Routed to: {route_taken}\n")

Unsafe mode is enabled. This allows execution of arbitrary code in the Jinja template. Use this only if you trust the source of the template.



Testing Query Router:

Query: How many vacation days do I get?
  → Routed to: hr_query

Query: Can I work remotely?
  → Routed to: hr_query

Query: What are the working hours?
  → Routed to: hr_query



### 3.2 Build Multi-Path RAG Pipeline

In [9]:
# Create specialized prompt templates for different query types
hr_template = """
You are an HR assistant. Answer the employee's HR-related question.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ query }}

Provide a clear, policy-based answer.
Answer:
"""

general_template = """
You are a helpful assistant. Answer the question based on the context.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""

print("✅ Created specialized templates for different query types")

✅ Created specialized templates for different query types


In [11]:
from haystack.utils import Secret

# Build routing pipeline (simplified example)
def create_routing_rag(query: str) -> str:
    """
    Simple routing RAG that chooses template based on query type.
    """
    # Determine route
    if any(word in query.lower() for word in ['leave', 'vacation', 'sick']):
        template = hr_template
        print(f"🔀 Routing to HR pipeline")
    else:
        template = general_template
        print(f"🔀 Routing to general pipeline")
    
    # Build and run pipeline
    pipeline = Pipeline()
    pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=company_store))
    pipeline.add_component("prompt", PromptBuilder(template=template))
    pipeline.add_component("llm", OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o"))
    
    pipeline.connect("retriever.documents", "prompt.documents")
    pipeline.connect("prompt", "llm")
    
    result = pipeline.run({
        "retriever": {"query": query, "top_k": 2},
        "prompt": {"query": query}
    })
    
    return result['llm']['replies'][0]

# Test routing
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    answer = create_routing_rag(query)
    print(f"Answer: {answer}")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Query: How many vacation days do I get?
🔀 Routing to HR pipeline


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Answer: According to our company policy, you are entitled to 20 days of annual leave each year. If you have any further questions about how to request your vacation days or manage your leave balances, please let me know!

Query: Can I work remotely?
🔀 Routing to general pipeline


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Answer: Yes, you can work remotely for up to 3 days per week, but you'll need to obtain approval from your manager.

Query: What are the working hours?
🔀 Routing to general pipeline
Answer: The standard working hours are from 9 AM to 5 PM, but flexible hours are available.


## 4. Web Search Fallback

When local knowledge is insufficient, fallback to web search.

### 4.1 Detect When to Use Web Search

First, determine if the local knowledge base has sufficient information.

In [12]:
def should_use_web_search(query: str, retrieved_docs: List[Document], threshold: float = 0.3) -> bool:
    """
    Determine if web search is needed based on retrieval confidence.
    
    Args:
        query: User query
        retrieved_docs: Documents from local retrieval
        threshold: Minimum score for confidence
        
    Returns:
        True if web search should be used
    """
    if not retrieved_docs:
        return True
    
    # Check if top document has low relevance score
    if retrieved_docs[0].score < threshold:
        return True
    
    # Could add more sophisticated checks:
    # - Query mentions current events/dates
    # - Query is about topics outside knowledge base domain
    
    return False

# Test with different queries
retriever = InMemoryBM25Retriever(document_store=company_store)

test_cases = [
    "How many vacation days do I get?",  # Should find in KB
    "What's the weather like today?",  # Not in KB
    "Who won the latest Nobel Prize?",  # Not in KB
]

print("Web Search Decision Logic:\n")
for query in test_cases:
    result = retriever.run(query=query, top_k=3)
    docs = result['documents']
    use_web = should_use_web_search(query, docs)
    
    print(f"Query: {query}")
    if docs:
        print(f"  Top doc score: {docs[0].score:.3f}")
    print(f"  Use web search: {'✅ Yes' if use_web else '❌ No'}\n")

Web Search Decision Logic:

Query: How many vacation days do I get?
  Top doc score: 2.273
  Use web search: ❌ No

Query: What's the weather like today?
  Use web search: ✅ Yes

Query: Who won the latest Nobel Prize?
  Use web search: ✅ Yes



### 4.2 Implement Web Fallback (Simulated)

Note: For actual web search, you'd use `SerperDevWebSearch` with an API key.

In [14]:
def rag_with_web_fallback(query: str) -> Dict[str, Any]:
    """
    RAG pipeline with web search fallback.
    Falls back to LLM's knowledge if web search unavailable.
    """
    # Try local retrieval first
    retriever = InMemoryBM25Retriever(document_store=company_store)
    retrieval_result = retriever.run(query=query, top_k=3)
    docs = retrieval_result['documents']
    
    use_web = should_use_web_search(query, docs, threshold=0.5)
    
    if use_web:
        print("🌐 Local knowledge insufficient - using web fallback")
        
        # If SerperDev API key available, use web search
        if SERPER_API_KEY:
            try:
                web_search = SerperDevWebSearch(api_key=Secret.from_env_var("SERPER_API_KEY"), top_k=3)
                web_result = web_search.run(query=query)
                
                # Convert web results to documents
                web_docs = []
                for item in web_result.get('documents', []):
                    web_docs.append(Document(
                        content=item.get('snippet', ''),
                        meta={'source': 'web', 'url': item.get('link', '')}
                    ))
                docs = web_docs
                source = "web_search"
            except Exception as e:
                print(f"  Web search failed: {e}")
                print("  Falling back to LLM general knowledge")
                docs = []
                source = "llm_knowledge"
        else:
            print("  No web search API configured")
            print("  Using LLM's general knowledge")
            docs = []
            source = "llm_knowledge"
    else:
        print("📚 Using local knowledge base")
        source = "local_kb"
    
    # Generate answer
    if docs:
        template = """
        Answer the question based on the provided information.
        
        Information:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Question: {{ query }}
        
        Answer:
        """
        prompt_builder = PromptBuilder(template=template)
        prompt = prompt_builder.run(documents=docs, query=query)['prompt']
    else:
        # No context, direct query
        prompt = f"Question: {query}\n\nAnswer:"
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    answer = llm.run(prompt=prompt)['replies'][0]
    
    return {
        'answer': answer,
        'source': source,
        'num_docs': len(docs),
        'docs': docs
    }

# Test fallback
test_queries = [
    "How many vacation days do I get?",
    "What's the weather like today?",
    "Explain quantum computing"
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    result = rag_with_web_fallback(query)
    
    print(f"\nSource: {result['source']}")
    print(f"Documents used: {result['num_docs']}")
    print(f"\nAnswer:\n{result['answer']}")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Query: How many vacation days do I get?
📚 Using local knowledge base

Source: local_kb
Documents used: 3

Answer:
You get 20 days of annual leave as vacation days.

Query: What's the weather like today?
🌐 Local knowledge insufficient - using web fallback
  Web search failed: An error occurred while querying SerperDevWebSearch. Error: Client error '400 Bad Request' for url 'https://google.serper.dev/search'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
  Falling back to LLM general knowledge

Source: llm_knowledge
Documents used: 0

Answer:
I'm sorry, but I can't provide real-time weather updates. To find out the current weather, I recommend checking a weather app or website such as Weather.com or a local news channel.

Query: Explain quantum computing
🌐 Local knowledge insufficient - using web fallback
  Web search failed: An error occurred while querying SerperDevWebSearch. Error: Client error '400 Bad Request' for url 'https://google.serper

## 5. Self-Correcting RAG (Self-RAG)

Self-RAG validates its own outputs and corrects if needed.

### 5.1 Relevance Assessment

First, check if retrieved documents are relevant to the query.

In [15]:
def assess_document_relevance(query: str, doc_content: str) -> Tuple[bool, str]:
    """
    Use LLM to assess if document is relevant to query.
    
    Returns:
        (is_relevant: bool, reasoning: str)
    """
    prompt = f"""
    Is the following document relevant for answering the query?
    
    Query: {query}
    
    Document: {doc_content}
    
    Respond with ONLY 'YES' or 'NO', followed by a brief reason.
    Format: YES/NO - [reason]
    """
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    response = llm.run(prompt=prompt)['replies'][0]
    
    is_relevant = response.strip().upper().startswith('YES')
    
    return is_relevant, response

# Test relevance assessment
print("Testing Relevance Assessment:\n")

query = "How many vacation days do employees get?"
relevant_doc = "Employees get 20 days annual leave."
irrelevant_doc = "The office is located at 123 Main Street."

is_rel, reason = assess_document_relevance(query, relevant_doc)
print(f"Query: {query}")
print(f"Doc: {relevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}\n")

is_rel, reason = assess_document_relevance(query, irrelevant_doc)
print(f"Doc: {irrelevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}")

Testing Relevance Assessment:

Query: How many vacation days do employees get?
Doc: Employees get 20 days annual leave.
Relevant: True
Reasoning: YES - The document states the number of vacation days employees get, which answers the query.

Doc: The office is located at 123 Main Street.
Relevant: False
Reasoning: NO - The document provides the office location, which is not relevant to the number of vacation days employees get.


### 5.2 Answer Validation

Check if the generated answer is grounded in the retrieved documents.

In [16]:
def validate_answer_grounding(answer: str, documents: List[Document]) -> Tuple[bool, str]:
    """
    Check if answer is grounded in provided documents.
    
    Returns:
        (is_grounded: bool, reasoning: str)
    """
    context = "\n".join([doc.content for doc in documents])
    
    prompt = f"""
    Is the following answer supported by the provided context?
    
    Context:
    {context}
    
    Answer:
    {answer}
    
    Respond with ONLY 'YES' if the answer is fully supported by the context, or 'NO' if it contains information not in the context or contradicts it.
    Format: YES/NO - [brief explanation]
    """
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    response = llm.run(prompt=prompt)['replies'][0]
    is_grounded = response.strip().upper().startswith('YES')
    
    return is_grounded, response

# Test answer validation
print("\nTesting Answer Validation:\n")

docs = [Document(content="Employees receive 20 days of annual vacation.")]
grounded_answer = "Employees get 20 vacation days per year."
hallucinated_answer = "Employees get 30 vacation days and unlimited sick leave."

is_valid, reason = validate_answer_grounding(grounded_answer, docs)
print(f"Answer: {grounded_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}\n")

is_valid, reason = validate_answer_grounding(hallucinated_answer, docs)
print(f"Answer: {hallucinated_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}")


Testing Answer Validation:

Answer: Employees get 20 vacation days per year.
Grounded: True
Reasoning: YES - The answer directly matches the information provided in the context.

Answer: Employees get 30 vacation days and unlimited sick leave.
Grounded: False
Reasoning: NO - The answer states that employees get 30 vacation days, but the context specifies that they receive 20 days of vacation. Additionally, unlimited sick leave is not mentioned in the context.


### 5.3 Self-Correcting Pipeline

In [17]:
def self_correcting_rag(query: str, max_iterations: int = 2) -> Dict[str, Any]:
    """
    Self-correcting RAG that validates and regenerates if needed.
    """
    print(f"\n🔍 Processing query: {query}\n")
    
    retriever = InMemoryBM25Retriever(document_store=company_store)
    
    for iteration in range(max_iterations):
        print(f"Iteration {iteration + 1}:")
        
        # Retrieve documents
        result = retriever.run(query=query, top_k=3)
        docs = result['documents']
        
        # Filter irrelevant documents
        relevant_docs = []
        for doc in docs:
            is_rel, reason = assess_document_relevance(query, doc.content)
            if is_rel:
                relevant_docs.append(doc)
                print(f"  ✅ Relevant doc: {doc.content[:60]}...")
            else:
                print(f"  ❌ Filtered out: {doc.content[:60]}...")
        
        if not relevant_docs:
            print("  ⚠️  No relevant documents found")
            return {
                'answer': "I don't have enough information to answer this question.",
                'grounded': False,
                'iterations': iteration + 1
            }
        
        # Generate answer
        template = """
        Answer the question based ONLY on the provided context. Do not add information not present in the context.
        
        Context:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Question: {{ query }}
        
        Answer:
        """
        
        prompt_builder = PromptBuilder(template=template)
        prompt = prompt_builder.run(documents=relevant_docs, query=query)['prompt']
        
        llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
        answer = llm.run(prompt=prompt)['replies'][0]
        
        print(f"  Generated answer: {answer[:100]}...")
        
        # Validate grounding
        is_grounded, validation = validate_answer_grounding(answer, relevant_docs)
        print(f"  Validation: {validation}")
        
        if is_grounded:
            print(f"  ✅ Answer is grounded!\n")
            return {
                'answer': answer,
                'grounded': True,
                'iterations': iteration + 1,
                'docs': relevant_docs
            }
        else:
            print(f"  ⚠️  Answer not fully grounded, retrying...\n")
    
    # Max iterations reached
    return {
        'answer': answer,
        'grounded': False,
        'iterations': max_iterations,
        'docs': relevant_docs,
        'warning': 'Could not generate fully grounded answer'
    }

# Test self-correcting RAG
result = self_correcting_rag("How many vacation days do employees receive?")
print("\nFinal Result:")
print(f"Answer: {result['answer']}")
print(f"Grounded: {result['grounded']}")
print(f"Iterations: {result['iterations']}")


🔍 Processing query: How many vacation days do employees receive?

Iteration 1:
  ✅ Relevant doc: Employees get 20 days annual leave. Sick leave is 10 days pe...
  ❌ Filtered out: Business expenses reimbursed within 30 days. Keep all receip...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


  ❌ Filtered out: Remote work allowed up to 3 days/week with manager approval....
  Generated answer: Employees receive 20 vacation days....
  Validation: YES - Employees receive 20 vacation days is supported by the context which states that employees get 20 days of annual leave.
  ✅ Answer is grounded!


Final Result:
Answer: Employees receive 20 vacation days.
Grounded: True
Iterations: 1


## 6. Conversational RAG with Memory

Track conversation history for natural multi-turn dialogue.

In [19]:
class ConversationMemory:
    """
    Simple conversation memory using a sliding window.
    """
    
    def __init__(self, max_messages: int = 10):
        self.max_messages = max_messages
        self.messages = deque(maxlen=max_messages)
    
    def add_user_message(self, content: str):
        """Add user message to history."""
        self.messages.append(ChatMessage.from_user(content))
    
    def add_assistant_message(self, content: str):
        """Add assistant message to history."""
        self.messages.append(ChatMessage.from_assistant(content))
    
    def get_messages(self) -> List[ChatMessage]:
        """Get all messages in history."""
        return list(self.messages)
    
    def get_context_string(self) -> str:
        """Get conversation history as formatted string."""
        context = []
        for msg in self.messages:
            role = "User" if msg.role == "user" else "Assistant"
            context.append(f"{role}: {msg.text}")
        return "\n".join(context)
    
    def clear(self):
        """Clear all history."""
        self.messages.clear()

# Test memory
memory = ConversationMemory(max_messages=6)
memory.add_user_message("Hello")
memory.add_assistant_message("Hi! How can I help?")
memory.add_user_message("What's the weather?")

print("Conversation History:\n")
print(memory.get_context_string())

Conversation History:

User: Hello
Assistant: Hi! How can I help?
User: What's the weather?


In [21]:
from haystack.components.generators.chat import OpenAIChatGenerator

In [22]:
class ConversationalRAG:
    """
    RAG system with conversation memory.
    """
    
    def __init__(self, document_store: InMemoryDocumentStore):
        self.document_store = document_store
        self.memory = ConversationMemory(max_messages=10)
        self.retriever = InMemoryBM25Retriever(document_store=document_store)
        self.llm = OpenAIChatGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    def query(self, user_message: str) -> str:
        """
        Process a query with conversation context.
        """
        # Add user message to memory
        self.memory.add_user_message(user_message)
        
        # Retrieve relevant documents
        retrieval_result = self.retriever.run(query=user_message, top_k=3)
        docs = retrieval_result['documents']
        
        # Build context from docs
        doc_context = "\n".join([doc.content for doc in docs])
        
        # Build messages for chat model
        messages = [
            ChatMessage.from_system(
                "You are a helpful assistant. Use the provided context and conversation history to answer questions."
            )
        ]
        
        # Add conversation history
        messages.extend(self.memory.get_messages()[:-1])  # Exclude last user message (will be added with context)
        
        # Add current query with context
        current_message = f"""
        Context from knowledge base:
        {doc_context}
        
        User question: {user_message}
        """
        messages.append(ChatMessage.from_user(current_message))
        
        # Generate response
        response = self.llm.run(messages=messages)
        assistant_message = response['replies'][0].text
        
        # Add to memory
        self.memory.add_assistant_message(assistant_message)
        
        return assistant_message
    
    def reset(self):
        """Reset conversation."""
        self.memory.clear()

# Test conversational RAG
conv_rag = ConversationalRAG(company_store)

print("\n" + "="*80)
print("CONVERSATIONAL RAG DEMO")
print("="*80)

conversation = [
    "How many vacation days do I get?",
    "What about sick days?",  # Follow-up that requires context
    "Can I work remotely?",
    "How many days per week?"  # Another follow-up
]

for user_msg in conversation:
    print(f"\n👤 User: {user_msg}")
    response = conv_rag.query(user_msg)
    print(f"🤖 Assistant: {response}")


CONVERSATIONAL RAG DEMO

👤 User: How many vacation days do I get?
🤖 Assistant: You get 20 days of annual leave for vacation.

👤 User: What about sick days?
🤖 Assistant: You receive 10 days of sick leave per year.

👤 User: Can I work remotely?
🤖 Assistant: Yes, you can work remotely for up to 3 days per week, but you will need approval from your manager.

👤 User: How many days per week?
🤖 Assistant: You can work remotely for up to 3 days per week, with manager approval.


## 7. Adaptive Retrieval Strategies

Adjust retrieval strategy based on query complexity.

In [23]:
def assess_query_complexity(query: str) -> str:
    """
    Classify query as simple, medium, or complex.
    
    Returns:
        'simple', 'medium', or 'complex'
    """
    prompt = f"""
    Classify the complexity of this query:
    
    Query: {query}
    
    Classification:
    - SIMPLE: Direct factual question, single piece of information
    - MEDIUM: Requires combining 2-3 pieces of information
    - COMPLEX: Multi-step reasoning, analysis, or comparison needed
    
    Respond with ONLY one word: SIMPLE, MEDIUM, or COMPLEX
    """
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    response = llm.run(prompt=prompt)['replies'][0].strip().upper()
    
    if 'SIMPLE' in response:
        return 'simple'
    elif 'COMPLEX' in response:
        return 'complex'
    else:
        return 'medium'

# Test query classification
test_queries = [
    "How many vacation days?",  # Simple
    "Compare remote work policy with vacation policy",  # Complex
    "What benefits are available?",  # Medium
]

print("Query Complexity Classification:\n")
for query in test_queries:
    complexity = assess_query_complexity(query)
    print(f"Query: {query}")
    print(f"Complexity: {complexity.upper()}\n")

Query Complexity Classification:

Query: How many vacation days?
Complexity: SIMPLE

Query: Compare remote work policy with vacation policy
Complexity: COMPLEX

Query: What benefits are available?
Complexity: SIMPLE



In [24]:
def adaptive_rag(query: str) -> Dict[str, Any]:
    """
    RAG that adapts strategy based on query complexity.
    
    Simple → Fast retrieval, fewer docs
    Medium → Standard retrieval
    Complex → More docs, deeper analysis
    """
    complexity = assess_query_complexity(query)
    print(f"\n🎯 Query complexity: {complexity.upper()}")
    
    # Adjust parameters based on complexity
    if complexity == 'simple':
        top_k = 1
        temperature = 0.1
        print("   Strategy: Fast retrieval (top_k=1, low temperature)")
    elif complexity == 'medium':
        top_k = 3
        temperature = 0.3
        print("   Strategy: Standard retrieval (top_k=3, medium temperature)")
    else:  # complex
        top_k = 5
        temperature = 0.5
        print("   Strategy: Deep retrieval (top_k=5, higher temperature for creativity)")
    
    # Retrieve
    retriever = InMemoryBM25Retriever(document_store=company_store)
    result = retriever.run(query=query, top_k=top_k)
    docs = result['documents']
    
    # Generate with appropriate template
    if complexity == 'complex':
        template = """
        Analyze the information and provide a comprehensive answer.
        
        Information:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Question: {{ query }}
        
        Provide a detailed answer with analysis:
        """
    else:
        template = """
        Answer concisely based on the information.
        
        Information:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Question: {{ query }}
        
        Answer:
        """
    
    prompt_builder = PromptBuilder(template=template)
    prompt = prompt_builder.run(documents=docs, query=query)['prompt']
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    answer = llm.run(prompt=prompt)['replies'][0]
    
    return {
        'answer': answer,
        'complexity': complexity,
        'docs_retrieved': len(docs),
        'temperature': temperature
    }

# Test adaptive RAG
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    result = adaptive_rag(query)
    
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nMetadata:")
    print(f"  Complexity: {result['complexity']}")
    print(f"  Docs retrieved: {result['docs_retrieved']}")
    print(f"  Temperature: {result['temperature']}")


Query: How many vacation days?


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🎯 Query complexity: SIMPLE
   Strategy: Fast retrieval (top_k=1, low temperature)

Answer:
20 days

Metadata:
  Complexity: simple
  Docs retrieved: 1
  Temperature: 0.1

Query: Compare remote work policy with vacation policy


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🎯 Query complexity: COMPLEX
   Strategy: Deep retrieval (top_k=5, higher temperature for creativity)

Answer:
The remote work policy and the vacation policy provide flexibility and benefits aimed at enhancing work-life balance for employees, but they differ in their structure and application.

### Remote Work Policy:
- **Flexibility**: Employees are allowed to work remotely up to three days a week, subject to managerial approval. This policy offers flexibility in terms of work location, which can be advantageous for employees seeking a balance between office presence and personal convenience.
- **Conditions**: The necessity for managerial approval means that employees may need to justify the necessity or benefit of working remotely and it might be subject to specific role requirements or performance metrics.
- **Regularity**: The policy allows for a regular, weekly occurrence, meaning employees can structure their work routines consistently, assuming managerial consent is obtained.

#

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



🎯 Query complexity: SIMPLE
   Strategy: Fast retrieval (top_k=1, low temperature)

Answer:
Flexible hours are available.

Metadata:
  Complexity: simple
  Docs retrieved: 1
  Temperature: 0.1


## 8. Comprehensive Agentic Pipeline

Combine all agentic features into one system.

In [25]:
class AgenticRAG:
    """
    Comprehensive agentic RAG system combining:
    - Adaptive retrieval
    - Web fallback
    - Self-correction
    - Conversation memory
    """
    
    def __init__(self, document_store: InMemoryDocumentStore):
        self.document_store = document_store
        self.memory = ConversationMemory()
        self.retriever = InMemoryBM25Retriever(document_store=document_store)
        
    def process_query(self, query: str, enable_web_fallback: bool = True) -> Dict[str, Any]:
        """
        Process query with full agentic capabilities.
        """
        print(f"\n{'='*80}")
        print(f"Processing: {query}")
        print(f"{'='*80}\n")
        
        # Step 1: Assess complexity
        complexity = assess_query_complexity(query)
        print(f"1️⃣  Complexity: {complexity.upper()}")
        
        # Step 2: Adaptive retrieval
        top_k = {'simple': 2, 'medium': 3, 'complex': 5}[complexity]
        print(f"2️⃣  Retrieving top {top_k} documents...")
        
        result = self.retriever.run(query=query, top_k=top_k)
        docs = result['documents']
        
        # Step 3: Check if web fallback needed
        use_web = enable_web_fallback and should_use_web_search(query, docs, threshold=0.4)
        
        if use_web:
            print("3️⃣  ⚠️  Low confidence - would use web fallback (simulated)")
            # In production, would call web search here
        else:
            print("3️⃣  ✅ Local knowledge sufficient")
        
        # Step 4: Generate with conversation context
        print("4️⃣  Generating response with conversation context...")
        
        template = """
        You are a helpful assistant. Answer based on the context and conversation history.
        
        Context:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Conversation history:
        {{ history }}
        
        Current question: {{ query }}
        
        Answer:
        """
        
        prompt_builder = PromptBuilder(template=template)
        prompt = prompt_builder.run(
            documents=docs,
            query=query,
            history=self.memory.get_context_string()
        )['prompt']
        
        llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
        answer = llm.run(prompt=prompt)['replies'][0]
        
        # Step 5: Validate grounding
        is_grounded, validation = validate_answer_grounding(answer, docs)
        print(f"5️⃣  Validation: {'✅ Grounded' if is_grounded else '⚠️  Not fully grounded'}")
        
        # Step 6: Update memory
        self.memory.add_user_message(query)
        self.memory.add_assistant_message(answer)
        print("6️⃣  Memory updated\n")
        
        return {
            'answer': answer,
            'metadata': {
                'complexity': complexity,
                'docs_used': len(docs),
                'grounded': is_grounded,
                'web_fallback_triggered': use_web
            }
        }

# Test comprehensive agentic RAG
agentic_rag = AgenticRAG(company_store)

test_conversation = [
    "How many vacation days do employees get?",
    "Can unused days be carried over?",
    "What about remote work?"
]

for query in test_conversation:
    result = agentic_rag.process_query(query)
    print(f"Answer: {result['answer']}")
    print(f"\nMetadata: {result['metadata']}")


Processing: How many vacation days do employees get?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ✅ Grounded
6️⃣  Memory updated

Answer: Employees get 20 days of annual leave.

Metadata: {'complexity': 'simple', 'docs_used': 2, 'grounded': True, 'web_fallback_triggered': False}

Processing: Can unused days be carried over?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ⚠️  Not fully grounded
6️⃣  Memory updated

Answer: The context provided does not specify whether unused vacation days can be carried over to the next year. To get accurate information on this, it would be best to refer to the company's employee handbook or consult the HR department.

Metadata: {'complexity': 'simple', 'docs_used': 2, 'grounded': False, 'web_fallback_triggered': False}

Processing: What about remote work?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ✅ Grounded
6️⃣  Memory updated

Answer: Remote work is allowed up to 3 days per week with manager approval.

Metadata: {'complexity': 'simple', 'docs_used': 2, 'grounded': True, 'web_fallback_triggered': False}


## 9. Evaluation & Metrics

Evaluate agentic RAG systems with specialized metrics.

In [26]:
# Evaluation metrics for agentic RAG
def evaluate_agentic_system(test_cases: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Evaluate agentic RAG system.
    
    Test cases should have:
    - query: str
    - expected_grounded: bool
    - expected_fallback: bool (optional)
    """
    results = []
    agentic_rag = AgenticRAG(company_store)
    
    for case in test_cases:
        query = case['query']
        result = agentic_rag.process_query(query, enable_web_fallback=True)
        
        results.append({
            'query': query,
            'grounded': result['metadata']['grounded'],
            'complexity': result['metadata']['complexity'],
            'docs_used': result['metadata']['docs_used'],
            'fallback_triggered': result['metadata']['web_fallback_triggered']
        })
    
    return pd.DataFrame(results)

# Test cases
test_cases = [
    {'query': 'How many vacation days?', 'expected_grounded': True},
    {'query': 'Can I work remotely?', 'expected_grounded': True},
    {'query': 'What is the weather today?', 'expected_grounded': False},
]

print("\nEvaluating Agentic RAG System...\n")
eval_df = evaluate_agentic_system(test_cases)
print(eval_df.to_string(index=False))

print(f"\n📊 Summary:")
print(f"  Grounded answers: {eval_df['grounded'].sum()} / {len(eval_df)}")
print(f"  Fallback triggered: {eval_df['fallback_triggered'].sum()} times")
print(f"  Avg docs used: {eval_df['docs_used'].mean():.1f}")


Evaluating Agentic RAG System...


Processing: How many vacation days?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ✅ Grounded
6️⃣  Memory updated


Processing: Can I work remotely?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ✅ Grounded
6️⃣  Memory updated


Processing: What is the weather today?



PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


1️⃣  Complexity: SIMPLE
2️⃣  Retrieving top 2 documents...
3️⃣  ✅ Local knowledge sufficient
4️⃣  Generating response with conversation context...
5️⃣  Validation: ⚠️  Not fully grounded
6️⃣  Memory updated

                     query  grounded complexity  docs_used  fallback_triggered
   How many vacation days?      True     simple          2               False
      Can I work remotely?      True     simple          2               False
What is the weather today?     False     simple          2               False

📊 Summary:
  Grounded answers: 2 / 3
  Fallback triggered: 0 times
  Avg docs used: 2.0


## 10. Exercises & Challenges

### Exercise 1: Implement Multi-Hop Reasoning

Create a system that can answer questions requiring information from multiple documents.

In [ ]:
# Your code here
# TODO: Implement multi-hop RAG
# Example query: "Compare vacation and sick leave policies"

### Exercise 2: Add Confidence Scores

Implement a confidence scoring system that rates answer quality 1-10.

In [ ]:
# Your code here
# TODO: Add confidence scoring
# Consider: document relevance, answer grounding, query complexity

### Challenge: Build Tool-Using Agent

Create an agent that can:
1. Search the knowledge base
2. Search the web
3. Perform calculations
4. Query a database (simulated)

The agent should decide which tools to use based on the query.

In [ ]:
# Your code here
# TODO: Implement tool-using agent with function calling

## 11. Summary & Next Steps

### What You've Learned

✅ **Conditional Routing**
- Route queries to specialized pipelines
- Dynamic decision-making based on query type

✅ **Web Search Fallback**
- Detect when local knowledge is insufficient
- Integrate external search sources

✅ **Self-Correcting RAG**
- Validate document relevance
- Check answer grounding
- Iterative refinement

✅ **Conversational RAG**
- Track conversation history
- Handle follow-up questions
- Context-aware responses

✅ **Adaptive Retrieval**
- Assess query complexity
- Adjust retrieval strategy dynamically
- Optimize for different scenarios

### Key Takeaways

1. **Intelligence through routing**: Make smart decisions, not just retrieve
2. **Validation is critical**: Self-correction prevents hallucinations
3. **Context matters**: Memory enables natural conversations
4. **Adaptability wins**: One size doesn't fit all queries
5. **Fallbacks are essential**: Local knowledge has limits

### What's Next?

In **Notebook 4 - Specialized RAG Techniques**, you'll learn:

- 🔧 **Corrective RAG (CRAG)** - Advanced self-correction patterns
- 🕸️ **GraphRAG** - Knowledge graph integration
- 📄 **Long-context RAG** - Handling extensive documents
- 🖼️ **Multimodal RAG** - Text + images together

### Additional Resources

- [Self-RAG Paper](https://arxiv.org/abs/2310.11511)
- [Corrective RAG Paper](https://arxiv.org/abs/2401.15884)
- [Adaptive RAG Research](https://arxiv.org/abs/2403.14403)

---

**Excellent progress!** You now have intelligent, agentic RAG systems. Continue to **Notebook 4** for cutting-edge specialized techniques! 🚀